# Consolidated PDP + Confirmed + Uncertain Development Wedge Engine

CSV-only notebook for JupyterLite.

**Inputs**
- `massive_PDP_production.csv`
- `massive_PDP_frcst.csv`
- `PDProutingexample.csv`: A Facility, B Well Name, E WI %
- `CFDroutingexample.csv`: A Facility, B Well Name, D WI %
- `Confirmed_Dev_Well_Production.csv`: horizontal repeating well blocks
- `UncertainDevRoutingExample.csv`: A Facility, B Well Name, C WI %, D TC Connection, E POP Date
- `MONTHLY_FORECAST_6COL.csv`: A TC/ENTITY_NAME, B Forecast Month, C Oil BBL/month, E Gas MCF/month

**Rules**
- Routing files are hard whitelists: source-file wells not routed are ignored.
- WI is converted from percent to decimal and applied at well-month level before aggregation.
- All PDP, confirmed-dev, and uncertain-dev wells are capped at **500 months**.
- PDP uses actual production through each well's last production month, then forecast.
- Confirmed dev starts its 500-month window at the first nonzero oil/gas month.
- Uncertain dev POP is rounded to the nearest first of month; Forecast Month 1 is assigned to that full month.

**Cases**
1. PDP Base
2. PDP + Confirmed Development
3. PDP + Confirmed + Uncertain Development

Each facility gets oil and gas rate+cumulative plots with peak rate, rate at 2/1/2034, and end cumulative annotations. Development cases show the incremental rate wedge. Exactly six monthly CSV outputs are written, one per case/fluid.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re, os, gc

PDP_PRODUCTION_FILE = "massive_PDP_production.csv"
PDP_FORECAST_FILE   = "massive_PDP_frcst.csv"
PDP_ROUTING_FILE    = "PDProutingexample.csv"
CFD_ROUTING_FILE    = "CFDroutingexample.csv"
CFD_PRODUCTION_FILE = "Confirmed_Dev_Well_Production.csv"
UNCERTAIN_ROUTING_FILE = "UncertainDevRoutingExample.csv"
UNCERTAIN_TC_FILE      = "MONTHLY_FORECAST_6COL.csv"

MAX_WELL_MONTHS = 500
ANNOTATION_DATE = pd.Timestamp("2034-02-01")
CHUNK_SIZE = 250_000

# Confirmed-dev horizontal layout: 6 data columns + 1 spacer
CFD_BLOCK_WIDTH = 7
CFD_DATA_START_ROW = 4
CFD_DATE_OFFSET = 0
CFD_OIL_OFFSET = 2
CFD_GAS_OFFSET = 4

SAVE_PNGS = True
PLOT_FOLDER = "Facility_Wedge_Plots"
if SAVE_PNGS:
    os.makedirs(PLOT_FOLDER, exist_ok=True)
print("Settings loaded")

In [ ]:
def normalize_name(v):
    if pd.isna(v): return ""
    return re.sub(r"\s+", " ", str(v).strip()).upper()

def normalize_tc(v):
    if pd.isna(v): return ""
    return re.sub(r"\s+", " ", str(v).strip()).upper()

def safe_filename(v):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(v).strip()).strip("_") or "Facility"

def nearest_first_of_month(v):
    if pd.isna(v): return pd.NaT
    dt=pd.Timestamp(v)
    this=pd.Timestamp(dt.year,dt.month,1)
    nxt=this+pd.DateOffset(months=1)
    return this if abs(dt-this) <= abs(nxt-dt) else nxt

def wi_decimal(s,label):
    x=pd.to_numeric(s,errors="coerce")
    if x.isna().any(): raise ValueError(f"{label}: blank/non-numeric WI")
    if ((x<0)|(x>100)).any(): raise ValueError(f"{label}: WI must be 0-100 percent")
    return x/100.0

def parse_cfd_dates(series):
    out=[]
    for v in series:
        if pd.isna(v) or str(v).strip()=="":
            out.append(pd.NaT); continue
        text=str(v).strip(); dt=pd.NaT
        for fmt in ("%y-%b","%b-%y"):
            if pd.isna(dt):
                try:
                    c=pd.to_datetime(text,format=fmt,errors="raise")
                    if 2000 <= c.year <= 2200: dt=c
                except: pass
        if pd.isna(dt):
            try:
                c=pd.to_datetime(text,errors="raise")
                if 2000 <= c.year <= 2200: dt=c
            except: pass
        if pd.isna(dt):
            try:
                n=float(v)
                if 30000 <= n <= 150000:
                    c=pd.Timestamp("1899-12-30")+pd.Timedelta(days=n)
                    if 2000 <= c.year <= 2200: dt=c
            except: pass
        if not pd.isna(dt): dt=pd.Timestamp(dt.year,dt.month,1)
        out.append(dt)
    return pd.Series(out,index=series.index,dtype="datetime64[ns]")

In [ ]:
# ROUTING FILES = HARD WHITELISTS

p=pd.read_csv(PDP_ROUTING_FILE)
if p.shape[1] < 5: raise ValueError("PDProutingexample.csv needs columns A-E")
pdp_route=p.iloc[:,[0,1,4]].copy(); pdp_route.columns=["Facility","Well_Name","WI_Pct"]
pdp_route["Facility"]=pdp_route["Facility"].astype(str).str.strip()
pdp_route["Well_Name"]=pdp_route["Well_Name"].astype(str).str.strip()
pdp_route["Match_Name"]=pdp_route["Well_Name"].apply(normalize_name)
pdp_route["WI"]=wi_decimal(pdp_route["WI_Pct"],"PDP routing")
pdp_route=pdp_route[(pdp_route["Facility"]!="")&(pdp_route["Match_Name"]!="")].copy()

c=pd.read_csv(CFD_ROUTING_FILE)
if c.shape[1] < 4: raise ValueError("CFDroutingexample.csv needs columns A-D")
cfd_route=c.iloc[:,[0,1,3]].copy(); cfd_route.columns=["Facility","Well_Name","WI_Pct"]
cfd_route["Facility"]=cfd_route["Facility"].astype(str).str.strip()
cfd_route["Well_Name"]=cfd_route["Well_Name"].astype(str).str.strip()
cfd_route["Match_Name"]=cfd_route["Well_Name"].apply(normalize_name)
cfd_route["WI"]=wi_decimal(cfd_route["WI_Pct"],"CFD routing")
cfd_route=cfd_route[(cfd_route["Facility"]!="")&(cfd_route["Match_Name"]!="")].copy()

u=pd.read_csv(UNCERTAIN_ROUTING_FILE)
if u.shape[1] < 5: raise ValueError("UncertainDevRoutingExample.csv needs columns A-E")
ud_route=u.iloc[:,[0,1,2,3,4]].copy(); ud_route.columns=["Facility","Well_Name","WI_Pct","TC_Connection","POP_Date"]
ud_route["Facility"]=ud_route["Facility"].astype(str).str.strip()
ud_route["Well_Name"]=ud_route["Well_Name"].astype(str).str.strip()
ud_route["Match_Name"]=ud_route["Well_Name"].apply(normalize_name)
ud_route["TC_Key"]=ud_route["TC_Connection"].apply(normalize_tc)
ud_route["WI"]=wi_decimal(ud_route["WI_Pct"],"Uncertain routing")
ud_route["POP_Date"]=pd.to_datetime(ud_route["POP_Date"],errors="coerce")
ud_route["Normalized_POP"]=ud_route["POP_Date"].apply(nearest_first_of_month)
ud_route=ud_route[(ud_route["Facility"]!="")&(ud_route["Match_Name"]!="")].copy()
if ud_route["TC_Key"].eq("").any() or ud_route["Normalized_POP"].isna().any():
    raise ValueError("Uncertain routing has blank TC or invalid POP date")

def check_unique(route,extra_cols,label):
    agg={"Facility":"nunique","WI":"nunique"}
    for x in extra_cols: agg[x]="nunique"
    chk=route.groupby("Match_Name").agg(agg)
    if (chk>1).any(axis=1).any():
        display(chk[(chk>1).any(axis=1)])
        raise ValueError(f"{label}: duplicated well has conflicting routing")

check_unique(pdp_route,[],"PDP")
check_unique(cfd_route,[],"CFD")
check_unique(ud_route,["TC_Key","Normalized_POP"],"Uncertain")
pdp_route=pdp_route.drop_duplicates("Match_Name").reset_index(drop=True)
cfd_route=cfd_route.drop_duplicates("Match_Name").reset_index(drop=True)
ud_route=ud_route.drop_duplicates("Match_Name").reset_index(drop=True)
PDP_ALLOWED=set(pdp_route["Match_Name"]); CFD_ALLOWED=set(cfd_route["Match_Name"]); UD_TCS=set(ud_route["TC_Key"])
print("PDP routed wells:",len(PDP_ALLOWED))
print("CFD routed wells:",len(CFD_ALLOWED))
print("Uncertain routed wells:",ud_route["Match_Name"].nunique())

In [ ]:
# LARGE PDP READER - only A/H/I/J, filtered in chunks
def read_large_pdp(filename):
    keep=[]
    for chunk in pd.read_csv(filename,usecols=[0,7,8,9],chunksize=CHUNK_SIZE):
        chunk.columns=["Well_Name","Date","Oil_BBL","Gas_MCF"]
        chunk["Match_Name"]=chunk["Well_Name"].apply(normalize_name)
        chunk=chunk[chunk["Match_Name"].isin(PDP_ALLOWED)].copy()
        if len(chunk)==0: continue
        chunk["Date"]=pd.to_datetime(chunk["Date"],errors="coerce").dt.to_period("M").dt.to_timestamp()
        chunk["Oil_BBL"]=pd.to_numeric(chunk["Oil_BBL"],errors="coerce").fillna(0)
        chunk["Gas_MCF"]=pd.to_numeric(chunk["Gas_MCF"],errors="coerce").fillna(0)
        chunk=chunk[chunk["Date"].notna()]
        keep.append(chunk.groupby(["Match_Name","Date"],as_index=False)[["Oil_BBL","Gas_MCF"]].sum())
    if not keep: return pd.DataFrame(columns=["Match_Name","Date","Oil_BBL","Gas_MCF"])
    d=pd.concat(keep,ignore_index=True)
    return d.groupby(["Match_Name","Date"],as_index=False)[["Oil_BBL","Gas_MCF"]].sum()

print("Loading PDP production..."); pdp_prod=read_large_pdp(PDP_PRODUCTION_FILE); gc.collect()
print("Loading PDP forecast..."); pdp_fcst=read_large_pdp(PDP_FORECAST_FILE); gc.collect()
print("PDP production wells used:",pdp_prod["Match_Name"].nunique())
print("PDP forecast wells used:",pdp_fcst["Match_Name"].nunique())

In [ ]:
# BUILD 500-MONTH NET PDP WELL PROFILES
def build_pdp_profiles():
    rows=[]; missing=[]
    for _,r in pdp_route.iterrows():
        key,fac,well,wi=r["Match_Name"],r["Facility"],r["Well_Name"],r["WI"]
        p=pdp_prod[pdp_prod["Match_Name"]==key].sort_values("Date").copy()
        f=pdp_fcst[pdp_fcst["Match_Name"]==key].sort_values("Date").copy()
        if len(p)==0 and len(f)==0:
            missing.append(well); continue
        start=p["Date"].min() if len(p) else f["Date"].min()
        prof=pd.DataFrame({"Date":pd.date_range(start=start,periods=MAX_WELL_MONTHS,freq="MS")})
        last_actual=p["Date"].max() if len(p) else pd.NaT
        pp=p[["Date","Oil_BBL","Gas_MCF"]].rename(columns={"Oil_BBL":"A_Oil","Gas_MCF":"A_Gas"})
        ff=f[["Date","Oil_BBL","Gas_MCF"]].rename(columns={"Oil_BBL":"F_Oil","Gas_MCF":"F_Gas"})
        prof=prof.merge(pp,on="Date",how="left").merge(ff,on="Date",how="left").fillna(0)
        if len(p): actual=prof["Date"]<=last_actual
        else: actual=pd.Series(False,index=prof.index)
        forecast=~actual
        prof["Gross_Oil_BBL"]=0.0; prof["Gross_Gas_MCF"]=0.0
        prof.loc[actual,"Gross_Oil_BBL"]=prof.loc[actual,"A_Oil"]
        prof.loc[actual,"Gross_Gas_MCF"]=prof.loc[actual,"A_Gas"]
        prof.loc[forecast,"Gross_Oil_BBL"]=prof.loc[forecast,"F_Oil"]
        prof.loc[forecast,"Gross_Gas_MCF"]=prof.loc[forecast,"F_Gas"]
        prof["Net_Oil_BBL"]=prof["Gross_Oil_BBL"]*wi
        prof["Net_Gas_MCF"]=prof["Gross_Gas_MCF"]*wi
        prof["Facility"]=fac; prof["Well_Name"]=well; prof["Match_Name"]=key
        rows.append(prof[["Facility","Well_Name","Match_Name","Date","Net_Oil_BBL","Net_Gas_MCF"]])
    if missing:
        print("WARNING routed PDP wells missing from both source files:")
        for w in missing: print(" ",w)
    if not rows: raise ValueError("No PDP profiles built")
    return pd.concat(rows,ignore_index=True)

def aggregate_component(d):
    if d is None or len(d)==0: return pd.DataFrame(columns=["Facility","Date","Net_Oil_BBL","Net_Gas_MCF"])
    return d.groupby(["Facility","Date"],as_index=False)[["Net_Oil_BBL","Net_Gas_MCF"]].sum().sort_values(["Facility","Date"])

pdp_wells=build_pdp_profiles(); pdp_fac=aggregate_component(pdp_wells); gc.collect()
print("PDP profiles built:",pdp_wells["Match_Name"].nunique())

In [ ]:
# CONFIRMED DEV - horizontal blocks, whitelist first, WI before aggregation
def build_cfd_profiles():
    raw=pd.read_csv(CFD_PRODUCTION_FILE,header=None)
    lookup=cfd_route.set_index("Match_Name")
    rows=[]; found=set()
    for start in range(0,raw.shape[1],CFD_BLOCK_WIDTH):
        if start+CFD_GAS_OFFSET >= raw.shape[1]: continue
        v=raw.iloc[0,start]
        if pd.isna(v): continue
        key=normalize_name(v)
        if key not in CFD_ALLOWED: continue
        found.add(key); rr=lookup.loc[key]
        t=pd.DataFrame({
            "Date":raw.iloc[CFD_DATA_START_ROW:,start+CFD_DATE_OFFSET].values,
            "Gross_Oil_BBL":raw.iloc[CFD_DATA_START_ROW:,start+CFD_OIL_OFFSET].values,
            "Gross_Gas_MCF":raw.iloc[CFD_DATA_START_ROW:,start+CFD_GAS_OFFSET].values})
        t["Date"]=parse_cfd_dates(t["Date"])
        t["Gross_Oil_BBL"]=pd.to_numeric(t["Gross_Oil_BBL"],errors="coerce").fillna(0)
        t["Gross_Gas_MCF"]=pd.to_numeric(t["Gross_Gas_MCF"],errors="coerce").fillna(0)
        t=t[t["Date"].notna()].groupby("Date",as_index=False)[["Gross_Oil_BBL","Gross_Gas_MCF"]].sum().sort_values("Date")
        nz=t[(t["Gross_Oil_BBL"]!=0)|(t["Gross_Gas_MCF"]!=0)]
        if len(nz)==0: continue
        first=nz["Date"].min(); end=first+pd.DateOffset(months=MAX_WELL_MONTHS-1)
        t=t[(t["Date"]>=first)&(t["Date"]<=end)].copy()
        t["Net_Oil_BBL"]=t["Gross_Oil_BBL"]*rr["WI"]
        t["Net_Gas_MCF"]=t["Gross_Gas_MCF"]*rr["WI"]
        t["Facility"]=rr["Facility"]; t["Well_Name"]=rr["Well_Name"]; t["Match_Name"]=key
        rows.append(t[["Facility","Well_Name","Match_Name","Date","Net_Oil_BBL","Net_Gas_MCF"]])
    missing=CFD_ALLOWED-found
    if missing:
        print("WARNING routed CFD wells not found in CFD production file:")
        for k in sorted(missing): print(" ",lookup.loc[k,"Well_Name"])
    return pd.concat(rows,ignore_index=True) if rows else pd.DataFrame(columns=["Facility","Well_Name","Match_Name","Date","Net_Oil_BBL","Net_Gas_MCF"])

cfd_wells=build_cfd_profiles(); cfd_fac=aggregate_component(cfd_wells); gc.collect()
print("CFD profiles built:",cfd_wells["Match_Name"].nunique())

In [ ]:
# UNCERTAIN DEV - TC monthly forecasts + routed WI/POP/facility
def read_tc_forecasts():
    keep=[]
    for chunk in pd.read_csv(UNCERTAIN_TC_FILE,usecols=[0,1,2,4],chunksize=CHUNK_SIZE):
        chunk.columns=["TC_Connection","Forecast_Month","Gross_Oil_BBL","Gross_Gas_MCF"]
        chunk["TC_Key"]=chunk["TC_Connection"].apply(normalize_tc)
        chunk=chunk[chunk["TC_Key"].isin(UD_TCS)].copy()
        if len(chunk)==0: continue
        chunk["Forecast_Month"]=pd.to_numeric(chunk["Forecast_Month"],errors="coerce")
        chunk["Gross_Oil_BBL"]=pd.to_numeric(chunk["Gross_Oil_BBL"],errors="coerce").fillna(0)
        chunk["Gross_Gas_MCF"]=pd.to_numeric(chunk["Gross_Gas_MCF"],errors="coerce").fillna(0)
        chunk=chunk[chunk["Forecast_Month"].notna()].copy(); chunk["Forecast_Month"]=chunk["Forecast_Month"].astype(int)
        chunk=chunk[(chunk["Forecast_Month"]>=1)&(chunk["Forecast_Month"]<=MAX_WELL_MONTHS)]
        keep.append(chunk.groupby(["TC_Key","Forecast_Month"],as_index=False)[["Gross_Oil_BBL","Gross_Gas_MCF"]].sum())
    if not keep: return pd.DataFrame(columns=["TC_Key","Forecast_Month","Gross_Oil_BBL","Gross_Gas_MCF"])
    d=pd.concat(keep,ignore_index=True)
    return d.groupby(["TC_Key","Forecast_Month"],as_index=False)[["Gross_Oil_BBL","Gross_Gas_MCF"]].sum()

def build_uncertain_profiles(tc):
    groups={k:g.sort_values("Forecast_Month").copy() for k,g in tc.groupby("TC_Key")}
    rows=[]; missing=[]
    for _,r in ud_route.iterrows():
        if r["TC_Key"] not in groups:
            missing.append((r["Well_Name"],r["TC_Connection"])); continue
        g=groups[r["TC_Key"]].copy()
        g["Date"]=g["Forecast_Month"].apply(lambda m:r["Normalized_POP"]+pd.DateOffset(months=int(m)-1))
        g["Net_Oil_BBL"]=g["Gross_Oil_BBL"]*r["WI"]
        g["Net_Gas_MCF"]=g["Gross_Gas_MCF"]*r["WI"]
        g["Facility"]=r["Facility"]; g["Well_Name"]=r["Well_Name"]; g["Match_Name"]=r["Match_Name"]
        rows.append(g[["Facility","Well_Name","Match_Name","Date","Net_Oil_BBL","Net_Gas_MCF"]])
    if missing:
        print("WARNING uncertain wells skipped because TC missing:")
        for w,tcname in missing: print(" ",w,"->",tcname)
    return pd.concat(rows,ignore_index=True) if rows else pd.DataFrame(columns=["Facility","Well_Name","Match_Name","Date","Net_Oil_BBL","Net_Gas_MCF"])

tc=read_tc_forecasts(); ud_wells=build_uncertain_profiles(tc); ud_fac=aggregate_component(ud_wells); gc.collect()
print("Uncertain profiles built:",ud_wells["Match_Name"].nunique())

In [ ]:
# BUILD THREE FACILITY-MONTH CASES
def merge_component(base,comp,prefix):
    c=comp.rename(columns={"Net_Oil_BBL":f"{prefix}_Oil_BBL","Net_Gas_MCF":f"{prefix}_Gas_MCF"})
    return base.merge(c,on=["Facility","Date"],how="outer")

def complete_calendar(d,cols):
    out=[]
    for fac,g in d.groupby("Facility"):
        cal=pd.DataFrame({"Date":pd.date_range(g["Date"].min(),g["Date"].max(),freq="MS")}); cal["Facility"]=fac
        cal=cal.merge(g,on=["Facility","Date"],how="left")
        for c in cols:
            if c not in cal: cal[c]=0.0
            cal[c]=pd.to_numeric(cal[c],errors="coerce").fillna(0)
        out.append(cal)
    return pd.concat(out,ignore_index=True)

def calc(d):
    d=d.sort_values(["Facility","Date"]).reset_index(drop=True).copy()
    d["Days_In_Month"]=d["Date"].dt.days_in_month
    d["Total_Oil_BPD"]=d["Total_Oil_BBL"]/d["Days_In_Month"]
    d["Total_Gas_MCFD"]=d["Total_Gas_MCF"]/d["Days_In_Month"]
    d["Total_Oil_Cumulative_BBL"]=d.groupby("Facility")["Total_Oil_BBL"].cumsum()
    d["Total_Gas_Cumulative_MCF"]=d.groupby("Facility")["Total_Gas_MCF"].cumsum()
    return d

case1=pdp_fac.rename(columns={"Net_Oil_BBL":"PDP_Oil_BBL","Net_Gas_MCF":"PDP_Gas_MCF"}).copy()
case1=complete_calendar(case1,["PDP_Oil_BBL","PDP_Gas_MCF"]); case1["Total_Oil_BBL"]=case1["PDP_Oil_BBL"]; case1["Total_Gas_MCF"]=case1["PDP_Gas_MCF"]; case1=calc(case1)

case2=merge_component(case1[["Facility","Date","PDP_Oil_BBL","PDP_Gas_MCF"]],cfd_fac,"Confirmed")
case2=complete_calendar(case2,["PDP_Oil_BBL","PDP_Gas_MCF","Confirmed_Oil_BBL","Confirmed_Gas_MCF"]); case2["Total_Oil_BBL"]=case2["PDP_Oil_BBL"]+case2["Confirmed_Oil_BBL"]; case2["Total_Gas_MCF"]=case2["PDP_Gas_MCF"]+case2["Confirmed_Gas_MCF"]; case2=calc(case2)

case3=merge_component(case2[["Facility","Date","PDP_Oil_BBL","PDP_Gas_MCF","Confirmed_Oil_BBL","Confirmed_Gas_MCF"]],ud_fac,"Uncertain")
case3=complete_calendar(case3,["PDP_Oil_BBL","PDP_Gas_MCF","Confirmed_Oil_BBL","Confirmed_Gas_MCF","Uncertain_Oil_BBL","Uncertain_Gas_MCF"]); case3["Total_Oil_BBL"]=case3["PDP_Oil_BBL"]+case3["Confirmed_Oil_BBL"]+case3["Uncertain_Oil_BBL"]; case3["Total_Gas_MCF"]=case3["PDP_Gas_MCF"]+case3["Confirmed_Gas_MCF"]+case3["Uncertain_Gas_MCF"]; case3=calc(case3)

# Add prior-case columns for wedge plots
def add_prior(total,prior):
    p=prior[["Facility","Date","Total_Oil_BPD","Total_Gas_MCFD","Total_Oil_Cumulative_BBL","Total_Gas_Cumulative_MCF"]].rename(columns={"Total_Oil_BPD":"Prior_Oil_BPD","Total_Gas_MCFD":"Prior_Gas_MCFD","Total_Oil_Cumulative_BBL":"Prior_Oil_Cumulative_BBL","Total_Gas_Cumulative_MCF":"Prior_Gas_Cumulative_MCF"})
    x=total.merge(p,on=["Facility","Date"],how="left")
    for c in ["Prior_Oil_BPD","Prior_Gas_MCFD","Prior_Oil_Cumulative_BBL","Prior_Gas_Cumulative_MCF"]: x[c]=x[c].fillna(0)
    return x
case2=add_prior(case2,case1); case3=add_prior(case3,case2)
print("Cases built")

In [ ]:
# PLOTS
def plot_profile(d,facility,label,stream,prior_label=None):
    g=d[d["Facility"]==facility].sort_values("Date").copy()
    if len(g)==0:return
    if stream=="Oil": rate,cum="Total_Oil_BPD","Total_Oil_Cumulative_BBL"; prate,pcum="Prior_Oil_BPD","Prior_Oil_Cumulative_BBL"; runit,cunit="BPD","BBL"
    else: rate,cum="Total_Gas_MCFD","Total_Gas_Cumulative_MCF"; prate,pcum="Prior_Gas_MCFD","Prior_Gas_Cumulative_MCF"; runit,cunit="MCF/D","MCF"
    x=list(g["Date"].dt.to_pydatetime()); y=g[rate].to_numpy(float); c=g[cum].to_numpy(float)
    peak=float(y.max()); row=g[g["Date"]==ANNOTATION_DATE]; r2034=float(row.iloc[0][rate]) if len(row) else np.nan; end=float(c[-1])
    fig,ax1=plt.subplots(figsize=(14,7)); ax2=ax1.twinx(); handles=[]; labels=[]
    if prior_label is None:
        l1,=ax1.plot(x,y,linewidth=2); l2,=ax2.plot(x,c,"--",linewidth=2); handles=[l1,l2]; labels=[f"{label} Rate",f"{label} Cumulative"]
    else:
        py=g[prate].to_numpy(float); pc=g[pcum].to_numpy(float)
        l0,=ax1.plot(x,py,linewidth=1.5); l1,=ax1.plot(x,y,linewidth=2); ax1.fill_between(x,py,y,alpha=.22)
        l2,=ax2.plot(x,pc,"--",linewidth=1.5); l3,=ax2.plot(x,c,"--",linewidth=2)
        handles=[l0,l1,l2,l3]; labels=[f"{prior_label} Rate",f"{label} Rate",f"{prior_label} Cumulative",f"{label} Cumulative"]
    rtxt=f"{r2034:,.0f} {runit}" if np.isfinite(r2034) else "N/A"
    ax1.text(.015,.97,f"Peak Rate: {peak:,.0f} {runit}\nRate on 2/1/2034: {rtxt}\nEnd Cumulative: {end:,.0f} {cunit}",transform=ax1.transAxes,va="top",bbox=dict(boxstyle="round",facecolor="white",alpha=.88))
    ax1.set_title(f"{facility} - {stream} - {label}"); ax1.set_xlabel("Date"); ax1.set_ylabel(f"{stream} Rate ({runit})"); ax2.set_ylabel(f"Cumulative {stream} ({cunit})"); ax1.grid(True,alpha=.3); ax1.legend(handles,labels,loc="best")
    if SAVE_PNGS: fig.savefig(os.path.join(PLOT_FOLDER,f"{safe_filename(label)}_{safe_filename(facility)}_{stream}.png"),dpi=160,bbox_inches="tight")
    plt.show()

for fac in sorted(case1["Facility"].dropna().unique()):
    plot_profile(case1,fac,"PDP Base","Oil"); plot_profile(case1,fac,"PDP Base","Gas")

cfd_affected=set(cfd_fac.loc[(cfd_fac["Net_Oil_BBL"]!=0)|(cfd_fac["Net_Gas_MCF"]!=0),"Facility"])
for fac in sorted(cfd_affected):
    plot_profile(case2,fac,"PDP + Confirmed Dev","Oil","PDP Base"); plot_profile(case2,fac,"PDP + Confirmed Dev","Gas","PDP Base")

ud_affected=set(ud_fac.loc[(ud_fac["Net_Oil_BBL"]!=0)|(ud_fac["Net_Gas_MCF"]!=0),"Facility"])
for fac in sorted(ud_affected):
    plot_profile(case3,fac,"PDP + Confirmed + Uncertain Dev","Oil","PDP + Confirmed Dev"); plot_profile(case3,fac,"PDP + Confirmed + Uncertain Dev","Gas","PDP + Confirmed Dev")

In [ ]:
# EXACTLY SIX CSV OUTPUTS
def oil_export(d,name):
    cols=["Facility","Date"]+[c for c in ["PDP_Oil_BBL","Confirmed_Oil_BBL","Uncertain_Oil_BBL"] if c in d.columns]+["Total_Oil_BBL","Total_Oil_BPD","Total_Oil_Cumulative_BBL"]
    x=d[cols].copy(); x.insert(0,"Case",name); return x
def gas_export(d,name):
    cols=["Facility","Date"]+[c for c in ["PDP_Gas_MCF","Confirmed_Gas_MCF","Uncertain_Gas_MCF"] if c in d.columns]+["Total_Gas_MCF","Total_Gas_MCFD","Total_Gas_Cumulative_MCF"]
    x=d[cols].copy(); x.insert(0,"Case",name); return x

oil_export(case1,"PDP Base").to_csv("01_PDP_Base_Oil.csv",index=False)
gas_export(case1,"PDP Base").to_csv("01_PDP_Base_Gas.csv",index=False)
oil_export(case2,"PDP + Confirmed Dev").to_csv("02_PDP_Plus_Confirmed_Oil.csv",index=False)
gas_export(case2,"PDP + Confirmed Dev").to_csv("02_PDP_Plus_Confirmed_Gas.csv",index=False)
oil_export(case3,"PDP + Confirmed + Uncertain Dev").to_csv("03_PDP_Plus_Confirmed_Plus_Uncertain_Oil.csv",index=False)
gas_export(case3,"PDP + Confirmed + Uncertain Dev").to_csv("03_PDP_Plus_Confirmed_Plus_Uncertain_Gas.csv",index=False)
print("Saved six requested CSVs")

In [ ]:
# FINAL ON-SCREEN QA SUMMARY (not saved as an extra CSV)
def qa(d,name):
    rows=[]
    for fac,g in d.groupby("Facility"):
        g=g.sort_values("Date"); oi=g["Total_Oil_BPD"].idxmax(); gi=g["Total_Gas_MCFD"].idxmax(); r=g[g["Date"]==ANNOTATION_DATE]
        rows.append({"Case":name,"Facility":fac,"Peak_Oil_BPD":g.loc[oi,"Total_Oil_BPD"],"Peak_Oil_Date":g.loc[oi,"Date"],"Peak_Gas_MCFD":g.loc[gi,"Total_Gas_MCFD"],"Peak_Gas_Date":g.loc[gi,"Date"],"Oil_BPD_2_1_2034":float(r.iloc[0]["Total_Oil_BPD"]) if len(r) else np.nan,"Gas_MCFD_2_1_2034":float(r.iloc[0]["Total_Gas_MCFD"]) if len(r) else np.nan,"End_Oil_Cum_BBL":g.iloc[-1]["Total_Oil_Cumulative_BBL"],"End_Gas_Cum_MCF":g.iloc[-1]["Total_Gas_Cumulative_MCF"]})
    return pd.DataFrame(rows)
qa_summary=pd.concat([qa(case1,"PDP Base"),qa(case2,"PDP + Confirmed Dev"),qa(case3,"PDP + Confirmed + Uncertain Dev")],ignore_index=True)
display(qa_summary)
print("500-month well-life cap applied to PDP, CFD, and uncertain dev")
print("All output volumes are net of WI")
print("Routing files are hard whitelists")